# ATT&CK Campaign Motivation Exploration

**Author:** Marc Labouchardiere (23857377) · **Supervisor:** Dr Jin B. Hong
**Date:** 2026-04-16 · **Thesis context:** MTDShield — adaptive MTD for dynamic networks.

## Why this notebook exists

Sub-problem 1 of the thesis requires generalised adversary profiles that differ by **motivation** (espionage vs financial vs destructive etc.), because motivation shapes TTP selection, persistence, and stealth requirements. MITRE ATT&CK is our TTP source — but ATT&CK Campaign and Intrusion-Set (Group) objects **do not populate** the STIX `primary_motivation` / `secondary_motivations` fields. Motivation exists only in free-text `description` prose.

This notebook:
1. **Proves the gap** programmatically (Part 1).
2. **Explores three motivation-attribution strategies**: description keyword extraction (Part 2), group-mediated inference via `attributed-to` relationships (Part 3), and unsupervised TTP clustering to test whether motivation-driven behaviour is discoverable from techniques alone (Part 4).
3. **Produces tactic-distribution profiles** (Part 5) usable as MTDShield adversary archetypes.

## Scope decisions

- **Self-contained**: no imports from `src/mtdsim/...`. Existing `enrich_group_profiles` / `extract_campaigns` helpers are deliberately not reused — exploratory analysis must be independent of prior implementation to avoid biasing findings.
- **Taxonomy**: STIX `attack-motivation-ov` (9 categories). Sparse at n≈30 but matches the spec; collapse only if data demands.
- **Staged delivery**: Part 1 runs in this first pass. Parts 2–6 are stubs to be filled in once Part 1 outputs inform the design (e.g. description length distribution shapes the keyword approach).

## Caveats up front

- n ≈ 25–35 campaigns → any clustering is noisy; exploratory, not statistical proof.
- CTI observation bias: multiple techniques per campaign often trace to a single vendor report.
- Ground truth must come from **external** CTI sources (not the ATT&CK descriptions themselves) to avoid circular validation.


---
## Part 1 — Schema audit: proving the gap

**Objective**: demonstrate programmatically that motivation is absent as structured data from ATT&CK Campaign and Group objects.


In [ ]:
# Cell 1 — Environment setup
from __future__ import annotations

import json
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mitreattack.stix20 import MitreAttackData

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)


In [ ]:
# Cell 2 — Load the ATT&CK Enterprise STIX bundle
BUNDLE_PATH = Path("enterprise-attack.json")
BUNDLE_URL = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json"

if not BUNDLE_PATH.exists():
    import urllib.request
    print(f"Downloading bundle to {BUNDLE_PATH} ...")
    urllib.request.urlretrieve(BUNDLE_URL, BUNDLE_PATH)

with BUNDLE_PATH.open() as f:
    raw_bundle = json.load(f)

# Find spec version from any object that advertises it
spec_versions = {obj.get("x_mitre_attack_spec_version") for obj in raw_bundle.get("objects", [])} - {None}
bundle_mtime = datetime.fromtimestamp(BUNDLE_PATH.stat().st_mtime).isoformat(timespec="seconds")

mitre = MitreAttackData(str(BUNDLE_PATH))

campaigns = mitre.get_campaigns()
groups = mitre.get_groups()

print(f"Bundle path:     {BUNDLE_PATH.resolve()}")
print(f"Bundle mtime:    {bundle_mtime}")
print(f"Spec version(s): {sorted(spec_versions)}")
print(f"Campaigns:       {len(campaigns)}")
print(f"Groups:          {len(groups)}")


In [ ]:
# Cell 3 — Campaign object audit: which STIX fields are actually populated?
MOTIVATION_FIELDS = ["primary_motivation", "secondary_motivations", "goals", "objective"]

def object_field_row(obj, name_attr="name"):
    props = {k: v for k, v in obj.items()} if hasattr(obj, "items") else dict(obj)
    desc = props.get("description", "") or ""
    row = {
        "stix_id": props.get("id"),
        "name": props.get(name_attr, ""),
        "n_keys": len(props),
        "desc_len": len(desc),
    }
    for field in MOTIVATION_FIELDS:
        row[f"has_{field}"] = field in props and bool(props.get(field))
    return row, sorted(props.keys())

campaign_rows = []
all_campaign_keys = set()
for c in campaigns:
    row, keys = object_field_row(c)
    campaign_rows.append(row)
    all_campaign_keys.update(keys)

campaign_df = pd.DataFrame(campaign_rows).sort_values("name").reset_index(drop=True)

print("Union of all keys observed across campaign objects:")
for k in sorted(all_campaign_keys):
    print(f"  - {k}")

print("\nMotivation-field population across campaigns:")
for field in MOTIVATION_FIELDS:
    col = f"has_{field}"
    print(f"  {field:25s}  populated in {campaign_df[col].sum():>3d} / {len(campaign_df)} campaigns")

campaign_df


In [ ]:
# Cell 3b — Print one full campaign object to make the absence concrete
sample = dict(campaigns[0])
print(f"Sample campaign: {sample.get('name')} ({sample.get('id')})")
print("-" * 72)
for k, v in sample.items():
    if k == "description":
        snippet = (v or "")[:240].replace("\n", " ")
        print(f"  {k}: {snippet}{'...' if len(v or '') > 240 else ''}")
    else:
        print(f"  {k}: {v!r}")


In [ ]:
# Cell 4 — Group (Intrusion-Set) object audit
group_rows = []
all_group_keys = set()
for g in groups:
    row, keys = object_field_row(g)
    group_rows.append(row)
    all_group_keys.update(keys)

group_df = pd.DataFrame(group_rows).sort_values("name").reset_index(drop=True)

print("Union of all keys observed across group (intrusion-set) objects:")
for k in sorted(all_group_keys):
    print(f"  - {k}")

print("\nMotivation-field population across groups:")
for field in MOTIVATION_FIELDS:
    col = f"has_{field}"
    n = group_df[col].sum()
    print(f"  {field:25s}  populated in {n:>3d} / {len(group_df)} groups")

# Any single positive case invalidates Parts 2–4's premise and must be flagged.
any_populated = group_df[[f"has_{f}" for f in MOTIVATION_FIELDS]].any().any()
if any_populated:
    print("\n!!  Premise check: at least one group carries a STIX motivation field — inspect before proceeding.")
else:
    print("\nPremise confirmed: no group in the bundle carries STIX motivation fields as structured data.")

group_df.head(10)


In [ ]:
# Cell 5 — Gap summary and description-length distribution
gap_summary = pd.DataFrame([
    {
        "object_type": "Campaign",
        "stix_supports_motivation": "objective (string)",
        "attck_populates_it": "No",
        "where_motivation_lives": "free-text `description`",
    },
    {
        "object_type": "Intrusion Set (Group)",
        "stix_supports_motivation": "primary_motivation, secondary_motivations, goals",
        "attck_populates_it": "No",
        "where_motivation_lives": "free-text `description`",
    },
    {
        "object_type": "Threat Actor",
        "stix_supports_motivation": "primary_motivation, secondary_motivations",
        "attck_populates_it": "ATT&CK does not use Threat Actor objects",
        "where_motivation_lives": "n/a",
    },
])

print("Description length summary:")
print("  Campaigns:", campaign_df["desc_len"].describe().round(0).to_dict())
print("  Groups:   ", group_df["desc_len"].describe().round(0).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
sns.histplot(campaign_df["desc_len"], bins=12, color="steelblue", ax=axes[0])
axes[0].set_title(f"Campaign description length (n={len(campaign_df)})")
axes[0].set_xlabel("characters")
sns.histplot(group_df["desc_len"], bins=20, color="coral", ax=axes[1])
axes[1].set_title(f"Group description length (n={len(group_df)})")
axes[1].set_xlabel("characters")
plt.tight_layout()
plt.show()

gap_summary


### Part 1 findings (to be filled in after running)

After executing Cells 1–5, record here:
- ATT&CK spec version and bundle date
- Campaign count (expect 25–35 for v16+)
- Group count (expect ~140 for v16+)
- Confirmation that no STIX motivation fields are populated
- Campaign description length distribution — this is the decisive input for Part 2's keyword approach viability
- Group description length distribution — compare against campaign; Part 3 should benefit from richer group text

**Decision point**: if median campaign description length is < 300 chars, keyword extraction (Part 2) will likely perform poorly and we should consider swapping it for an LLM-based extraction or demoting Part 2 to a pure baseline.


---
## Part 2 — Description-based motivation extraction *(stub — implement after Part 1 review)*

**Plan summary** (see `/home/marc/.claude/plans/encapsulated-fluttering-llama.md` for the approved full plan):

- **Cell 6**: STIX 9-category taxonomy (`attack-motivation-ov`) as `dict[category, list[regex]]`. Use `\b...\b` word boundaries. Drop "reconn" from espionage keywords (recon is universal).
- **Cell 7**: Apply to campaign descriptions. Abstain threshold ≥ 2 keyword hits → else `unknown`.
- **Cell 8**: Apply to descriptions of `attributed-to` groups. Combined decision rule: agreement → confident; disagreement → prefer group-text, flag conflict.
- **Cell 9**: External ground truth built from CTI sources **outside** ATT&CK (MITRE group-page references, vendor reports, MISP notes). Cite each label's source.
- **Cell 10**: Confusion matrix, per-category precision/recall. Baseline, not target.


In [ ]:
# Cell 6 — stub (Part 2). Implement after Part 1 findings.


In [ ]:
# Cell 7 — stub (Part 2). Implement after Part 1 findings.


In [ ]:
# Cell 8 — stub (Part 2). Implement after Part 1 findings.


In [ ]:
# Cell 9 — stub (Part 2). Implement after Part 1 findings.


In [ ]:
# Cell 10 — stub (Part 2). Implement after Part 1 findings.


---
## Part 3 — Group-mediated motivation inference *(stub)*

- **Cell 11**: Build campaign→group(s) attribution map via `mitre.get_groups_attributing_to_campaign(campaign_stix_id)`. Networkx bipartite viz. Flag campaigns with zero attribution.
- **Cell 12**: Curated `GROUP_MOTIVATIONS` dict restricted to groups attributed to ≥1 campaign. STIX 9-cat labels. Each entry cites its CTI source in a comment.
- **Cell 13**: Propagate group→campaign. Multi-group conflict rule: majority, else `ambiguous`. Compare vs. Cell 9 ground truth.


In [ ]:
# Cell 11 — stub (Part 3). Implement after Parts 1–2.


In [ ]:
# Cell 12 — stub (Part 3). Implement after Parts 1–2.


In [ ]:
# Cell 13 — stub (Part 3). Implement after Parts 1–2.


---
## Part 4 — TTP-based clustering *(stub)*

- **Cell 14**: Build (a) binary campaign×technique matrix, (b) count campaign×tactic (14-dim) matrix via `mitre.get_techniques_used_by_campaign(...)`.
- **Cell 15**: Sparsity stats. **Do not** pad with group-inherited techniques (leaks the attribution label).
- **Cell 16**: PCA + UMAP 2D scatter, coloured by Cell 9 ground truth.
- **Cell 17**: **Primary** — Ward hierarchical on tactic-level (k=3,4,5). **Secondary** — hierarchical with Jaccard on technique-level. Silhouette, dendrogram. Explicit n≈30 caveat.
- **Cell 18**: ARI / NMI vs ground truth, with 1000× label-permutation test for honest null.
- **Cell 19**: Characterise each cluster: top tactics, distinctive techniques (chi²), mean technique count, temporal span. Name archetypes.


In [ ]:
# Cell 14 — stub (Part 4). Implement after Parts 1–3.


In [ ]:
# Cell 15 — stub (Part 4). Implement after Parts 1–3.


In [ ]:
# Cell 16 — stub (Part 4). Implement after Parts 1–3.


In [ ]:
# Cell 17 — stub (Part 4). Implement after Parts 1–3.


In [ ]:
# Cell 18 — stub (Part 4). Implement after Parts 1–3.


In [ ]:
# Cell 19 — stub (Part 4). Implement after Parts 1–3.


---
## Part 5 — Tactic distribution profiles *(stub)*

- **Cell 20**: Per-motivation normalised tactic distributions using the best-performing label source from Parts 2/3/4.
- **Cell 21**: Heatmap (motivation × tactic) + radar chart overlay.
- **Cell 22**: Export to `notebooks/gap_out/campaign_motivations_YYYY-MM-DD.csv` and `tactic_profiles_YYYY-MM-DD.json`. CSV: `campaign_id, name, motivation_label, label_source, confidence`.


In [ ]:
# Cell 20 — stub (Part 5). Implement after Part 4.


In [ ]:
# Cell 21 — stub (Part 5). Implement after Part 4.


In [ ]:
# Cell 22 — stub (Part 5). Implement after Part 4.


---
## Part 6 — Summary and recommendations *(stub — Cell 23)*

To be written after Parts 1–5 complete. Five findings:
1. Whether the data gap is confirmed (Part 1).
2. Keyword extraction performance vs ground truth (Part 2).
3. Group-mediated inference reliability (Part 3).
4. Whether TTP clusters align with motivation (Part 4) — either direction is a valid thesis finding.
5. Recommendation for MTDShield: use tactic distribution profiles from Part 5 to parameterise adversary archetypes.
